In [17]:
!pip install deap --quiet

In [18]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# Define bounds for each variable as (min, max)
BOUNDS_LOW = [-10, -5, -2]   # Minimum bounds for variables
BOUNDS_HIGH = [10,  5, 10]   # Maximum bounds for variables

# Create a fitness and individual class
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimize objective
creator.create("Individual", list, fitness=creator.FitnessMin)

# Initialize DEAP toolbox
toolbox = base.Toolbox()

# Attribute generator that initializes within the variable bounds
def create_individual():
    return [random.uniform(low, high) for low, high in zip(BOUNDS_LOW, BOUNDS_HIGH)]

# Register individual and population creation functions
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Define an objective function
def evaluate(individual):
    # Example: Sphere function (just a simple sum of squares)
    return sum(x**2 for x in individual),

toolbox.register("evaluate", evaluate)

# Crossover and mutation operators
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selBest)

# Ensure that mutated individuals stay within bounds
def check_bounds(individual):
    for i in range(len(individual)):
        individual[i] = np.clip(individual[i], BOUNDS_LOW[i], BOUNDS_HIGH[i])
    return individual, 0 # Return individual and penalty (0 in this case, since repaired)

# Decorate the mutation operator to enforce bounds
# toolbox.decorate("mutate", tools.DeltaPenalty(lambda ind: min(ind) >= min(BOUNDS_LOW) and max(ind) <= max(BOUNDS_HIGH), 0))
toolbox.decorate("mutate", tools.DeltaPenalty(check_bounds, 0))  # Use check_bounds function

def printBest(popul, s):
  # Get the best individual
  best_ind = tools.selBest(popul[0], 1)[0]
  print(f"{s} Strategy --- \nBest individual: {best_ind}  \
        \nBest fitness: {best_ind.fitness.values[0]}")

pop_size = 20 # Population size

# Run the evolutionary algorithm
def esPlus():
    population = toolbox.population(n=pop_size)
    final_population = algorithms.eaMuPlusLambda(
        population,
        toolbox,
        mu=pop_size,
        lambda_=50,
        cxpb=0.7,
        mutpb=0.2,
        ngen=20,
        verbose=False
    )
    printBest(final_population, "Plus ")

def esComma():
    population = toolbox.population(n=pop_size)
    final_population = algorithms.eaMuCommaLambda(
        population,
        toolbox,
        mu=pop_size,
        lambda_=50,
        cxpb=0.7,
        mutpb=0.2,
        ngen=500,
        verbose=False
    )
    printBest(final_population, "Comma")

if __name__ == "__main__":
    esPlus()
    esComma()

Plus  Strategy --- 
Best individual: [np.float64(-0.0011444467736648838), np.float64(0.005812373744431281), np.float64(0.0013412641142217239)]          
Best fitness: 3.689243638680507e-05
Comma Strategy --- 
Best individual: [np.float64(-1.437695820577524e-11), np.float64(4.7527883672509977e-14), np.float64(-2.6933060674302e-12)]          
Best fitness: 2.1395308372319073e-22
